### titanic 예측
* titanic_cleaning.csv 파일을 불러와서 진행하시오.
* PassengerId는 순차적인 번호이므로 제외하고 진행하시오
* train, test 80:20으로 진행하시오
* 여러가지 모델을 생성 후 최적의 모델을 이용하시오
* 최적의 모델을 찾고 자신이 탑승했을 경우 인적사항을 넣고 결과를 예측하시오
>
* 컬럼
    - PassengerId : 고객 번호
    - Survived : 0이면 사망, 1이면 생존
    - Pclass : 티켓 등급 : 1, 2, 3
    - Sex : 성별, 0.남성, 1.여성
    - Age : 나이
    - SibSp : 형제, 자매, 배우자의 합
    - Parch : 부모, 자식의 합
    - Fare : 요금

In [1]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('../data_set/4.분류/titanic_cleaning.csv')
df.head()

,PassengerId,Survived,Pclass,Sex,Age,SibSp,Parch,Fare
0,1,0,3,0,22.0,1,0,7.2500
1,2,1,1,1,38.0,1,0,71.2833
2,3,1,3,1,26.0,0,0,7.9250
3,4,1,1,1,35.0,1,0,53.1000
4,5,0,3,0,35.0,0,0,8.0500


### feature, label 분리

In [2]:
df.columns

Index(['PassengerId', 'Survived', 'Pclass', 'Sex', 'Age', 'SibSp', 'Parch',
       'Fare'],
      dtype='object')

In [3]:
features = [ 'Pclass', 'Sex', 'Age', 'SibSp', 'Parch','Fare']
label = 'Survived'
X =df[features]
y = df[label]

In [4]:
# 훈련, 테스트 셋 분리
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = \
                            train_test_split(df[features], df[label], test_size=0.2)
print('총 개수 : ',X.shape, y.shape)
print('train 개수 : ',X_train.shape, y_train.shape)
print('test 개수 : ',X_test.shape, y_test.shape)

총 개수 :  (891, 6) (891,)
train 개수 :  (712, 6) (712,)
test 개수 :  (179, 6) (179,)


In [5]:
from sklearn.neighbors import KNeighborsClassifier
kn = KNeighborsClassifier()
kn.fit(X_train, y_train)
kn.score(X_test,y_test)

0.6927374301675978

In [6]:
import sklearn.svm as svm
svm_linear = svm.SVC(kernel='linear')
svm_linear.fit(X_train, y_train)
svm_linear.score(X_test, y_test)

0.7877094972067039

In [7]:
svm_rbf = svm.SVC(kernel='rbf')
svm_rbf.fit(X_train, y_train)
svm_rbf.score(X_test, y_test)

0.6759776536312849

In [8]:
from sklearn.tree import DecisionTreeClassifier
#모델 얻어오기
dt_clf = DecisionTreeClassifier()
#학습을 통한 모델 생성
dt_clf.fit(X_train, y_train)
dt_clf.score(X_test, y_test)

0.7653631284916201

In [9]:
svm_rbf = svm.SVC(kernel='rbf', probability=True)
svm_rbf.fit(X_train, y_train)

kn = KNeighborsClassifier()
kn.fit(X_train, y_train)

dt_clf = DecisionTreeClassifier()
dt_clf.fit(X_train, y_train)

,criterion,'gini'
,splitter,'best'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,None
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,class_weight,None


In [10]:
from sklearn.ensemble import VotingClassifier
vo = VotingClassifier(estimators=[('svc',svm_rbf),('knn',kn),('DecisionTree',dt_clf)] , voting='soft' )
vo.fit(X_train, y_train)
print('svm rbf : ', svm_rbf.score(X_test, y_test))
print('kn ', kn.score(X_test, y_test))
print('dt_clf ', dt_clf.score(X_test, y_test))
print('vo : ', vo.score(X_test, y_test))

svm rbf :  0.6759776536312849
kn  0.6927374301675978
dt_clf  0.770949720670391
vo :  0.776536312849162


In [11]:
from sklearn.ensemble import RandomForestClassifier
rfc = RandomForestClassifier()
rfc.fit(X_train, y_train)
rfc.score(X_test,y_test)

0.8268156424581006

In [12]:
from sklearn.ensemble import GradientBoostingClassifier
gbc = GradientBoostingClassifier()
gbc.fit(X_train, y_train)
gbc.score(X_test,y_test)

0.8491620111731844

### 생존, 죽음 여부 확인
* Survived : 0이면 사망, 1이면 생존
* Pclass : 티켓 등급 : 1, 2, 3
* Sex : 성별, 0.남성, 1.여성
* Age : 나이
* SibSp : 형제, 자매, 배우자의 합
* Parch : 부모, 자식의 합
* Fare : 요금

In [13]:
#['Pclass','Sex','Age','SibSp','Parch','Fare']
test = [[1, 0, 35, 1, 1, 10]]
result = rfc.predict(test)
result

array([0])

In [14]:
if(result[0] == 1):
    print('생존에 성공 하셨습니다!!!')
else:
    print("아고야~즐거운 하루 되세요~^^")

아고야~즐거운 하루 되세요~^^


In [16]:
import joblib
joblib.dump( rfc, "rfc_model.joblib")
print("모델 저장 완료")

모델 저장 완료


In [17]:
# 모델 로드
model = joblib.load("rfc_model.joblib")
"모델 로드 완료"

'모델 로드 완료'

In [18]:
test = [[1, 0, 35, 1, 1, 10]]
result = model.predict(test)
result

array([0])

In [19]:
!pip list

Package                           Version
--------------------------------- --------------------
aiobotocore                       2.25.0
aiodns                            3.5.0
aiohappyeyeballs                  2.6.1
aiohttp                           3.13.2
aioitertools                      0.12.0
aiosignal                         1.4.0
alabaster                         0.7.16
altair                            5.5.0
anaconda-anon-usage               0.7.5
anaconda-auth                     0.12.3
anaconda-catalogs                 0.2.0
anaconda-cli-base                 0.7.0
anaconda-client                   1.14.0
anaconda-navigator                2.7.0
anaconda-project                  0.11.1
annotated-types                   0.6.0
anyio                             4.10.0
appdirs                           1.4.4
archspec                          0.2.5
argon2-cffi                       21.3.0
argon2-cffi-bindings              25.1.0
arrow                             1.4.0
astroid      

In [ ]:
joblib==1.5.2
scikit-learn==1.7.2

In [ ]:
pip install joblib==1.5.2 scikit-learn==1.7.2